# S05 — Maamoura Phase 2 delta and parent-checkpoint audit

This focused validation audit documents the executed Maamoura delta and Phase 1 parent-checkpoint comparisons. It is retained as supplementary provenance rather than as a main pipeline notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, importlib.util, json, sys

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
EXPERIMENT = PROJECT / "Ablations" / "Maamoura_Phase2_Delta_Parent_Factorial_Seed42"
RUN_ROOT = EXPERIMENT / "runs"
PREDICTION_ROOT = EXPERIMENT / "test_predictions"
OUTPUT_ROOT = EXPERIMENT / "article_plots"
for path in (RUN_ROOT, PREDICTION_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

BASE_RUNNER = PROJECT / "Ablations" / "Phase2_Hypothesis_Validation" / "runtime" / "run_phase2_hypothesis_ablation.py"
ENGINE_PATH = PROJECT / "Source" / "Project" / "low_canopy_growthloss_ablation_runner.py"
ADAPTER_PATH = PROJECT / "Source" / "Project" / "aoi_masked_phase2_adapter.py"
CONFIG_PATH = PROJECT / "Source" / "Project" / "b4_c15_config.py"

PHASE1_RUN = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Ablations\Phase1_Sampler_Cursor_Ablation\runs\maamoura\final_catalog\MAAMOURA_PHASE1_NATURAL_SAMPLING_V2_SEED42")
P1_ANY = PHASE1_RUN / "checkpoints" / "best_any.ckpt"
P1_SLOPE = PHASE1_RUN / "checkpoints" / "best_slope.ckpt"

PRODUCTION_P2 = PROJECT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "maamoura" / "MAAMOURA_B4_C15_CTRL_GL000_D2_K2_HD3_SEED42"
SLOPE_DELTA3_P2 = PROJECT / "Ablations" / "Uniform_Checkpoint_Comparison_Seed42" / "runs" / "maamoura" / "training" / "MAAMOURA_B4_C15_UNIFORM_P1_BEST_SLOPE_P2_BEST_COMPROMISE_SEED42_SEED42"

SEED = 42
EXPECTED_TEST_N = 1799
RUN_TRAINING = True
AUTHORIZE_TEST_EVALUATION = True

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

base = load_module("maamoura_factorial_base", BASE_RUNNER)
cfgmod = load_module("maamoura_factorial_config", CONFIG_PATH)
print("Experiment:", EXPERIMENT)


In [ ]:
for required in (P1_ANY, P1_SLOPE, BASE_RUNNER, ENGINE_PATH, ADAPTER_PATH, CONFIG_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)

p1_config_path = PHASE1_RUN / "artifacts" / "run_config.json"
if not p1_config_path.is_file():
    raise FileNotFoundError(p1_config_path)
p1_args = json.loads(p1_config_path.read_text(encoding="utf-8"))["args"]
p1_checks = {
    "seed_42": int(p1_args["seed"]) == 42,
    "natural_sampling": p1_args["train_sampler_mode"] == "natural",
    "phase1_huber_delta_1": float(p1_args["huber_beta"]) == 1.0,
    "test_disabled": not bool(p1_args["eval_test_at_end"]),
}
if not all(p1_checks.values()):
    raise RuntimeError(f"Phase 1 lineage mismatch: {p1_checks}")

VARIANTS = {
    "best_any__p2_delta3": {"parent_rule": "best_any", "parent": P1_ANY, "p2_delta": 3.0, "run_dir": PRODUCTION_P2, "reuse": True},
    "best_slope__p2_delta3": {"parent_rule": "best_slope", "parent": P1_SLOPE, "p2_delta": 3.0, "run_dir": SLOPE_DELTA3_P2, "reuse": True},
    "best_any__p2_delta1": {"parent_rule": "best_any", "parent": P1_ANY, "p2_delta": 1.0, "run_dir": RUN_ROOT / "best_any__p2_delta1", "reuse": False},
    "best_slope__p2_delta1": {"parent_rule": "best_slope", "parent": P1_SLOPE, "p2_delta": 1.0, "run_dir": RUN_ROOT / "best_slope__p2_delta1", "reuse": False},
}

for name, variant in VARIANTS.items():
    variant["parent_sha256"] = file_sha256(variant["parent"])
    variant["checkpoint"] = variant["run_dir"] / "checkpoints" / "best_compromise.ckpt"

design = pd.DataFrame([
    {"variant": name, "phase1_delta": 1.0, "phase1_parent": v["parent_rule"],
     "phase1_sha256": v["parent_sha256"], "phase2_delta": v["p2_delta"],
     "existing_checkpoint": v["checkpoint"].is_file(), "run_dir": str(v["run_dir"])}
    for name, v in VARIANTS.items()
])
design.to_csv(EXPERIMENT / "00_prespecified_factorial_design.csv", index=False)
display(design)
print("Phase 1 checks:", p1_checks)


In [ ]:
def prepare_variant(name, variant):
    engine = load_module(f"factorial_engine_{name}", ENGINE_PATH)
    adapter = load_module(f"factorial_adapter_{name}", ADAPTER_PATH)
    cfg = engine.FORESTS["maamoura"]
    cfg["parent"] = variant["parent"]
    cfg["parent_sha"] = variant["parent_sha256"]
    cfg["ablation_family"] = "Maamoura_Phase2_Delta_Parent_Factorial_Seed42"
    candidate_spec = {"drop_m": 2.0, "K": 2, "lambda_growth": 0.0, "huber_delta": variant["p2_delta"]}
    engine.LOW_CANOPY_CANDIDATES = {name: candidate_spec}
    engine.roots = lambda _forest: (variant["run_dir"].parent, EXPERIMENT / "reports" / name)
    modules = adapter.prepare_aoi_masked_modules(engine)
    return engine, modules, cfg, candidate_spec

def validate_existing(name, variant):
    checkpoint = variant["checkpoint"]
    config_path = variant["run_dir"] / "config.json"
    if not checkpoint.is_file() or not config_path.is_file():
        return False
    saved = json.loads(config_path.read_text(encoding="utf-8"))
    growth = saved["growth_loss_provenance"]
    checks = {
        "parent_sha": saved["phase1_parent"]["source_sha256"] == variant["parent_sha256"],
        "seed": int(saved["seed"]) == SEED,
        "delta": float(saved["huber_delta"]) == variant["p2_delta"],
        "D": float(growth["persistent_drop_m"]) == 2.0,
        "K": int(growth["persistent_required_consecutive_flags"]) == 2,
        "lambda_temp": float(saved["lambda_growth"]) == 0.0,
    }
    if not all(checks.values()):
        raise RuntimeError(f"{name}: incompatible existing run: {checks}")
    print("REUSE", name, checkpoint, flush=True)
    return True

def train_variant(name, variant):
    engine, modules, cfg, spec = prepare_variant(name, variant)
    shots, records = base.harmonize(engine, modules, "maamoura")
    train_ds = modules["B4SequenceCropDataset"](
        records["train"], shots, crop_size=96, samples_per_epoch=264,
        drop_channels=(), seed=SEED, center_on_gedi=True,
        balanced_height_anchors=True, height_bins=cfg["height_bins"],
    )
    val_ds = modules["B4SequenceCropDataset"](
        records["val"], shots, crop_size=96,
        samples_per_epoch=max(132, 4 * len(records["val"])),
        drop_channels=(), seed=SEED + 10000, center_on_gedi=True,
        balanced_height_anchors=False, height_bins=cfg["height_bins"],
    )
    model = engine.fresh_model(cfg, modules)
    modules["train"](
        model=model, train_dataset=train_ds, val_dataset=val_ds,
        official_repo=engine.OFFICIAL_REPO, phase1_checkpoint=cfg["parent"],
        run_dir=variant["run_dir"], device=engine.DEVICE, seed=SEED,
        batch_size=1, max_steps=cfg["max_steps"], val_every_steps=66,
        patience_evals=20, learning_rate=1e-4, weight_decay=5e-3,
        lambda_growth=0.0, lambda_slope=0, lambda_std=0, lambda_bias=0,
        lambda_anti_zero=0, supervised_loss_name="huber",
        huber_delta=variant["p2_delta"], height_weight_mode="none",
        slope_min=0, slope_max=2, disturbance_indicator=-1,
        disturbance_rule="persistent_running_max", persistent_drop_m=2.0,
        persistent_required_consecutive_flags=2, full_disturbance_window=True,
        min_height=cfg["eval_min"], max_height=cfg["eval_max"],
        checkpoint_min_slope=0, checkpoint_min_std_ratio=0,
        checkpoint_max_std_ratio=2, checkpoint_max_abs_bias=5,
        warmup_cycles=3, plateau_patience=8, plateau_factor=0.5,
        lr_min=1e-6, grad_clip=1, allow_resume=True, reuse_completed=True,
    )
    del model, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for name, variant in VARIANTS.items():
    complete = validate_existing(name, variant)
    if not complete:
        if variant["reuse"]:
            raise FileNotFoundError(f"Required reusable run missing: {variant['run_dir']}")
        if not RUN_TRAINING:
            raise RuntimeError(f"Training disabled but {name} is missing")
        print("TRAIN", name, flush=True)
        train_variant(name, variant)
        if not validate_existing(name, variant):
            raise RuntimeError(f"{name}: training ended without a valid best_compromise checkpoint")


In [ ]:
registry_rows = []
for name, variant in VARIANTS.items():
    checkpoint = variant["checkpoint"]
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    metrics = state.get("metrics", {})
    registry_rows.append({
        "variant": name, "phase1_delta": 1.0,
        "phase1_parent_rule": variant["parent_rule"],
        "phase1_checkpoint": str(variant["parent"]),
        "phase1_sha256": variant["parent_sha256"],
        "phase2_delta": variant["p2_delta"],
        "phase2_checkpoint_rule": "best_compromise.ckpt selected on VAL only",
        "phase2_checkpoint": str(checkpoint), "phase2_sha256": file_sha256(checkpoint),
        "selection_split": "VAL", "test_used_for_selection": False,
        **{f"val_{key}": metrics.get(key) for key in ("n", "mae", "rmse", "r2", "bias", "slope", "std_ratio")},
    })
registry = pd.DataFrame(registry_rows)
registry.to_csv(EXPERIMENT / "01_frozen_checkpoint_registry.csv", index=False)
display(registry)
if len(registry) != 4 or not registry["phase2_checkpoint"].map(lambda p: Path(p).is_file()).all():
    raise RuntimeError("TEST blocked: the four-cell registry is incomplete")
print("[PASS] Four VAL-selected checkpoints frozen before TEST.")


In [ ]:
def support_fingerprint(ids):
    return hashlib.sha256("\n".join(sorted(map(str, ids))).encode("utf-8")).hexdigest()

def evaluate_variant(row):
    name = row.variant
    variant = VARIANTS[name]
    engine, modules, cfg, _ = prepare_variant(name, variant)
    checkpoint = Path(row.phase2_checkpoint)
    if file_sha256(checkpoint) != row.phase2_sha256:
        raise RuntimeError(f"{name}: checkpoint hash changed")
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    _, shots, records = engine.build_data("maamoura", modules, include_test=True)
    model = engine.fresh_model(cfg, modules)
    model.prediction_head.load_state_dict(state["prediction_head"], strict=True)
    model.eval()
    _, nearest = modules["evaluate_full_patch_temporal_nearest"](
        model=model, records=records["test"], shots=shots, device=engine.DEVICE,
        split="test", drop_channels=(), min_height=cfg["eval_min"],
        max_height=cfg["eval_max"], progress_every=1,
    )
    nearest["aux_shot_uid"] = nearest["aux_shot_uid"].astype(str)
    if len(nearest) != EXPECTED_TEST_N or not nearest["aux_shot_uid"].is_unique:
        raise RuntimeError(f"{name}: unexpected TEST support")
    nearest["split"] = "test"
    nearest["variant"] = name
    nearest["checkpoint_sha256"] = row.phase2_sha256
    out_dir = PREDICTION_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / "test_unique_nearest.csv.gz"
    nearest.to_csv(out, index=False, compression="gzip")
    lineage = {
        "variant": name, "n": len(nearest),
        "support_sha256": support_fingerprint(nearest["aux_shot_uid"]),
        "phase1_delta": 1.0, "phase1_parent_rule": variant["parent_rule"],
        "phase1_parent_sha256": variant["parent_sha256"],
        "phase2_delta": variant["p2_delta"], "phase2_rule": "best_compromise.ckpt",
        "phase2_sha256": row.phase2_sha256, "selection_split": "VAL",
        "evaluation_split": "TEST",
    }
    (out_dir / "lineage.json").write_text(json.dumps(lineage, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out

if AUTHORIZE_TEST_EVALUATION:
    prediction_paths = [evaluate_variant(row) for row in registry.itertuples(index=False)]
    supports = [json.loads((p.parent / "lineage.json").read_text(encoding="utf-8"))["support_sha256"] for p in prediction_paths]
    if len(set(supports)) != 1:
        raise RuntimeError("The four variants do not use the identical TEST support")
    print("[PASS] Identical TEST support across all four variants:", supports[0])


In [ ]:
SITES = {
    name: {
        "ecosystem": "Maamoura_factorial",
        "input": PREDICTION_ROOT / name / "test_unique_nearest.csv.gz",
        "eval_min": 2.0, "eval_max": 20.0,
        "height_bins": np.asarray([0, 5, 10, 15, 20], float),
        "expected_n": EXPECTED_TEST_N,
    }
    for name in VARIANTS
}


In [ ]:
factorial_metrics = pd.DataFrame([run_forest_report(name) for name in VARIANTS])
factorial_metrics["phase1_delta"] = 1.0
factorial_metrics["phase1_parent"] = factorial_metrics["forest"].map(lambda n: VARIANTS[n]["parent_rule"])
factorial_metrics["phase2_delta"] = factorial_metrics["forest"].map(lambda n: VARIANTS[n]["p2_delta"])
factorial_metrics.to_csv(EXPERIMENT / "02_factorial_test_metrics.csv", index=False)
display(factorial_metrics[["forest", "phase1_parent", "phase2_delta", "n", "mae", "rmse", "r2", "bias", "slope", "corr", "std_ratio"]])


In [ ]:
# Descriptive contrasts only; no TEST-guided checkpoint selection is performed.
factorial_metrics = pd.read_csv(EXPERIMENT / '02_factorial_test_metrics.csv')
m = factorial_metrics.set_index('forest')
rows = []
for parent in ('best_any', 'best_slope'):
    d1, d3 = f'{parent}__p2_delta1', f'{parent}__p2_delta3'
    rows.append({'contrast': f'Phase 2 delta 1 minus 3 | parent={parent}',
                 **{f'delta_{k}': m.loc[d1, k] - m.loc[d3, k] for k in ('mae', 'rmse', 'r2', 'bias', 'slope', 'std_ratio')}})
for delta in (1, 3):
    slope, any_ = f'best_slope__p2_delta{delta}', f'best_any__p2_delta{delta}'
    rows.append({'contrast': f'best_slope minus best_any | Phase 2 delta={delta}',
                 **{f'delta_{k}': m.loc[slope, k] - m.loc[any_, k] for k in ('mae', 'rmse', 'r2', 'bias', 'slope', 'std_ratio')}})
contrasts = pd.DataFrame(rows)
contrasts.to_csv(EXPERIMENT / '03_factorial_contrasts.csv', index=False)
display(contrasts)
decision = factorial_metrics.copy()
decision['abs_slope_from_1'] = (decision['slope'] - 1).abs()
decision['abs_std_ratio_from_1'] = (decision['std_ratio'] - 1).abs()
decision.to_csv(EXPERIMENT / '04_descriptive_metrics_no_model_selection.csv', index=False)
display(decision[['forest', 'mae', 'rmse', 'r2', 'slope', 'std_ratio', 'abs_slope_from_1', 'abs_std_ratio_from_1']])
print('Audit complete. Do not use TEST results to launch or select additional variants.')
